# SupremeAI - Gemma 2B Inference Server

Runs Gemma 2B-it on Colab and serves OpenAI-compatible API for SupremeAI.

### Requirements:
1. Runtime > Change runtime type > T4 GPU
2. HuggingFace Token - https://huggingface.co/settings/tokens


In [ ]:
# Step 1: Install dependencies (CLEAN INSTALL)
!pip uninstall -y transformers optimum accelerate bitsandbytes airllm flask flask-cors sentence-transformers -q 2>/dev/null || true
!pip install -q transformers==4.41.2 accelerate==0.28.0 bitsandbytes==0.42.0
!pip install -q optimum
!pip install -q airllm
!pip install -q flask==3.0.0 flask-cors==4.0.0

# Install Cloudflare Tunnel
!curl -L --output cloudflared.tgz https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.tgz 2>/dev/null
!tar -xzf cloudflared.tgz 2>/dev/null
!chmod +x cloudflared

import torch
import sys
print('\n' + '='*70)
print('✓ Dependencies installed successfully')
print('='*70)
print(f'  Python: {sys.version.split()[0]}')
print(f'  PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')
else:
    print('  ⚠️  NO GPU - Change Runtime to T4!')
print(f'  Cloudflare Tunnel: installed ✓')
print('='*70)

In [ ]:
# Step 2: Configure tokens + Clear cache - Gemma 2B (Token Set 1)
import os
import shutil

HF_TOKEN = 'hf_PASTE_YOUR_TOKEN_HERE'  # Replace with your HF token

MODEL_ID = 'google/gemma-2b-it'
COMPRESSION = '4bit'
MAX_NEW_TOKENS = 512
PORT = 8081

print('\n' + '='*70)
print('🔧 Configuration')
print('='*70)
print(f'Model: Gemma 2B-it')
print(f'Model ID: {MODEL_ID}')
print(f'Compression: {COMPRESSION}')
print(f'Max Tokens: {MAX_NEW_TOKENS}')
print(f'HF Token: {HF_TOKEN[:20]}...')
print('='*70)

# Clear HuggingFace cache to free space
print('\nClearing cache for disk space...')
cache_dir = os.path.expanduser('~/.cache/huggingface/hub')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir, ignore_errors=True)
    print('✓ HuggingFace cache cleared')

# Create HF token config
os.environ['HF_TOKEN'] = HF_TOKEN
print('✓ HF_TOKEN configured')
print('='*70)

In [ ]:
# Step 3: Load Gemma 2B model
import time

print('\n' + '='*70)
print('📥 Loading Gemma 2B model...')
print('='*70)
start = time.time()

try:
    print('Attempting to load with AirLLM...')
    from airllm import AutoModel
    kw = {'compression': COMPRESSION}
    if HF_TOKEN.startswith('hf_') and len(HF_TOKEN) > 10:
        kw['hf_token'] = HF_TOKEN
    
    model = AutoModel.from_pretrained(MODEL_ID, **kw)
    elapsed = time.time() - start
    print(f'✓ AirLLM loaded successfully in {elapsed:.0f}s')
    
except Exception as e:
    print(f'⚠️  AirLLM warning: {str(e)[:100]}')
    print('Falling back to transformers...')
    
    from transformers import AutoModelForCausalLM, AutoTokenizer
    token = HF_TOKEN if HF_TOKEN.startswith('hf_') else None
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        token=token,
        device_map='auto',
        load_in_4bit=True if COMPRESSION == '4bit' else False
    )
    model.tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
    elapsed = time.time() - start
    print(f'✓ Transformers loaded successfully in {elapsed:.0f}s')

print('='*70)
print(f'✓ Model ready: {MODEL_ID}')
print('='*70)

In [ ]:
# Step 4: Start Flask API + Cloudflare Tunnel
import uuid, time, torch, subprocess, threading, sys
from flask import Flask, request, jsonify
from flask_cors import CORS
from datetime import datetime

app = Flask(__name__)
CORS(app)
stats = {'req': 0, 'err': 0, 'start': time.time()}

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'healthy',
        'provider': 'cloudflare-tunnel',
        'model_active': MODEL_ID,
        'model_name': 'Gemma 2B-it',
        'compression': COMPRESSION,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'uptime': int(time.time() - stats['start']),
        'requests_total': stats['req'],
        'errors_total': stats['err']
    })

@app.route('/v1/models', methods=['GET'])
def list_models():
    return jsonify({
        'object': 'list',
        'data': [{
            'id': MODEL_ID,
            'object': 'model',
            'owned_by': 'supremeai-cloudflare',
            'name': 'Gemma 2B-it'
        }]
    })

@app.route('/v1/chat/completions', methods=['POST'])
def chat():
    try:
        stats['req'] += 1
        data = request.json
        msgs = data.get('messages', [])
        mt = min(data.get('max_tokens', MAX_NEW_TOKENS), MAX_NEW_TOKENS)
        
        parts = []
        for m in msgs:
            r, c = m.get('role', 'user'), m.get('content', '')
            if r == 'system':
                parts.append('[INST] <<SYS>>\n' + c + '\n<</SYS>>\n')
            elif r == 'user':
                parts.append('[INST] ' + c + ' [/INST]')
            elif r == 'assistant':
                parts.append(c)
        
        prompt = '\n'.join(parts)
        
        toks = model.tokenizer(
            [prompt],
            return_tensors='pt',
            return_attention_mask=False,
            truncation=True,
            max_length=2048,
            padding=False
        )
        il = len(toks['input_ids'][0])
        gen = model.generate(
            toks['input_ids'].cuda(),
            max_new_tokens=mt,
            use_cache=True,
            return_dict_in_generate=True,
            do_sample=False
        )
        nt = gen.sequences[0][il:]
        txt = model.tokenizer.decode(nt, skip_special_tokens=True).strip()
        
        return jsonify({
            'id': 'chatcmpl-' + uuid.uuid4().hex[:12],
            'object': 'chat.completion',
            'created': int(time.time()),
            'model': MODEL_ID,
            'choices': [{
                'index': 0,
                'message': {'role': 'assistant', 'content': txt},
                'finish_reason': 'stop'
            }],
            'usage': {
                'prompt_tokens': int(il),
                'completion_tokens': int(len(nt)),
                'total_tokens': int(il + len(nt))
            }
        })
    except Exception as e:
        stats['err'] += 1
        return jsonify({
            'error': {
                'message': str(e),
                'type': 'server_error',
                'code': 500
            }
        }), 500

# Start Flask in background thread
def run_flask():
    app.run(host='0.0.0.0', port=PORT, threaded=True)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

print('\n' + '='*70)
print('🚀 SupremeAI Gemma 2B Server STARTING')
print('='*70)
print(f'Model: Gemma 2B-it')
print(f'Compression: {COMPRESSION} | Max Tokens: {MAX_NEW_TOKENS}')
print(f'Local API: http://localhost:{PORT}')
print('='*70)
print('Starting Cloudflare Tunnel...')
print('This will provide a public HTTPS URL\n')

# Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1
)

# Wait for tunnel to start
time.sleep(4)

print('\n' + '='*70)
print('✓ Cloudflare Tunnel ACTIVE')
print('='*70)
print('\nYour public API URL will be displayed in the output below')
print('Format: https://xxxx-xxxx-xxxx.trycloudflare.com')
print('\nEndpoints:')
print('  POST   /v1/chat/completions  (Send messages)')
print('  GET    /health               (Health check)')
print('  GET    /v1/models            (List models)')
print('\n' + '='*70)
print('⚠️  KEEP THIS NOTEBOOK RUNNING!')
print('Cross-browser tab closure will break the tunnel')
print('='*70 + '\n')

# Keep the tunnel alive
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print('\n\nShutting down...')
    tunnel_proc.terminate()